# NullVector Unified Postgres Cookbook (Specs 01-09)

This notebook replaces the older PostgreSQL cookbooks with one end-to-end walkthrough of the current NullVector surface.

Prerequisites:
- A reachable PostgreSQL database exposed through `NULLVECTOR_POSTGRES_CONNINFO`
- Optional live LLM access via `OPENROUTER_API_KEY` and `NULLVECTOR_LLM_MODEL`
- If no live LLM adapter is available, the notebook falls back to a deterministic noop gateway for the optional summarized-tree path

The notebook is safe to rerun because it uses deterministic run ids.


## Section 1 — Setup And Environment

This cookbook is Postgres-backed, but it keeps the runnable path deterministic where possible. The only optional gateway usage is the summarized Markdown tree path; everything else can run with the library's deterministic fallbacks.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from collections import Counter
from pathlib import Path


def banner(title: str) -> None:
    print()
    print("=" * 60)
    print(title)
    print("=" * 60)


def show_json(title: str, payload: object) -> None:
    banner(title)
    print(json.dumps(payload, indent=2, sort_keys=True, default=str))


ROOT = Path.cwd()
print(f"Repository root: {ROOT}")
print(f"Python executable: {sys.executable}")


## Section 2 — Imports

The imports below intentionally use the current public surface that landed with Specs 01-09, plus the public storage helpers needed to derive PostgreSQL artifact refs.


In [ ]:
from nullvector.domain import (
    AcquisitionRequest,
    DescriptionSelectionRequest,
    DocumentDescriptionRecord,
    DocumentDescriptionRequest,
    DocumentFilterClause,
    DocumentFilterOperator,
    DocumentMetadataRecord,
    DocumentPrefilterRequest,
    DocumentSemanticProxySource,
    MetadataSelectionPlan,
    MetadataSelectionRequest,
    NodeCard,
    PreferenceAwareTreeSearchRequest,
    PreferenceScope,
    PreferenceSnippet,
    SourceDocumentKind,
    TreeBuildRequest,
    TreeCompactionRequest,
    TreeCompactionSettings,
    TreeSearchRequest,
)
from nullvector.ingest.acquisition_service import AcquisitionService
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayService,
    NoopProviderAdapter,
    NoopScriptedResponse,
)
from nullvector.retrieval import (
    DescriptionSelectionService,
    DocumentDescriptionBuilder,
    DocumentSemanticProxyBuilder,
    MetadataSelectionService,
    PreferenceAwareTreeSearchService,
    QueryPlanner,
    RetrievalCorpusBuilder,
    RetrievalQAService,
    RetrievalRanker,
    RetrievalService,
    SemanticPrefilterService,
    TreeSearchService,
    load_document_description,
    load_document_description_manifest,
    load_retrieval_corpus,
)
from nullvector.storage import (
    PostgresStorageConfig,
    build_document_store,
    build_postgres_artifact_ref,
)
from nullvector.tree import (
    TreeCompactionService,
    build_tree,
    expand_serving_node_ids_to_canonical_node_ids,
    load_compacted_node_mappings,
    load_compacted_tree,
)

try:
    from nullvector.llm.adapters import LiteLLMAdapter
except ImportError as exc:  # litellm is an optional dependency
    LiteLLMAdapter = None
    LITELLM_IMPORT_ERROR = exc
else:
    LITELLM_IMPORT_ERROR = None

## Section 3 — Configuration

We keep two demo documents in play:
- a committed PDF fixture from `fixtures/pdfs/phase01/`
- a Markdown file written into a local notebook temp directory so Spec 05 is part of the same walkthrough


In [ ]:
COOKBOOK_TMP_ROOT = ROOT / "cookbook" / "_tmp" / "postgres_unified"
COOKBOOK_TMP_ROOT.mkdir(parents=True, exist_ok=True)

PDF_SOURCE_PATH = ROOT / "fixtures" / "pdfs" / "phase01" / "born_digital_with_outline.pdf"
MARKDOWN_SOURCE_PATH = COOKBOOK_TMP_ROOT / "spec05_operating_handbook.md"
MARKDOWN_SOURCE_TEXT = "\n".join(
    (
        "# Operating Handbook",
        "Handbook overview for the quarterly operating playbook.",
        "",
        "## Revenue Policies",
        "Revenue policies describe recognition guardrails and margin reviews.",
        "",
        "## Litigation Policies",
        "Litigation policies describe case deadlines, filing checkpoints, and escalation paths.",
        "",
        "## Customer Support",
        "Customer support runbooks explain triage and escalation routing.",
        "",
        "## Controls Checklist",
        "Controls require approvals, ownership checks, and handoff reviews.",
        "",
        "## Appendix Notes",
        "Appendix notes contain glossary entries and reference links.",
    )
)
MARKDOWN_SOURCE_PATH.write_text(MARKDOWN_SOURCE_TEXT + "\n", encoding="utf-8")

POSTGRES_CONNINFO = os.environ.get(
    "NULLVECTOR_POSTGRES_CONNINFO",
    "postgresql://REDACTED_DB_CRED@localhost:5432/nullvector",
)
POSTGRES_SCHEMA = os.environ.get("NULLVECTOR_POSTGRES_SCHEMA", "public")
DEFAULT_MODEL = os.environ.get("NULLVECTOR_LLM_MODEL", "openrouter/openai/gpt-4.1-mini")
COLLECTION_ID = "cookbook-postgres-demo"

RUN_IDS = {
    "pdf_acquisition": "cookbook-pdf-acquisition",
    "pdf_tree": "cookbook-pdf-tree",
    "pdf_retrieval": "cookbook-pdf-retrieval",
    "pdf_description": "cookbook-pdf-description",
    "markdown_acquisition": "cookbook-md-acquisition",
    "markdown_tree": "cookbook-md-tree",
    "markdown_retrieval": "cookbook-md-retrieval",
    "markdown_description": "cookbook-md-description",
    "markdown_compaction": "cookbook-md-compaction",
    "metadata_selection": "cookbook-metadata-selection",
    "description_selection": "cookbook-description-selection",
    "semantic_prefilter": "cookbook-semantic-prefilter",
    "tree_search": "cookbook-tree-search",
    "preference_tree_search": "cookbook-preference-tree-search",
}

pg_config = PostgresStorageConfig(conninfo=POSTGRES_CONNINFO, schema=POSTGRES_SCHEMA)
store = build_document_store(pg_config)


def manifest_ref(*, run_type: str, run_id: str, document_id: str) -> str:
    return build_postgres_artifact_ref(
        run_type=run_type,
        run_id=run_id,
        document_id=document_id,
        artifact_path="manifest.json",
    )


def load_node_cards_from_tree_manifest(tree_manifest) -> tuple[NodeCard, ...]:
    if tree_manifest.node_cards_path is None:
        return ()
    payload = store.read_json_artifact(tree_manifest.node_cards_path)
    return tuple(NodeCard.model_validate(item) for item in payload)


def node_title_map(tree_manifest) -> dict[str, str]:
    return {
        card.node_id: card.title
        for card in load_node_cards_from_tree_manifest(tree_manifest)
    }


def build_demo_gateway() -> tuple[GatewayService, str]:
    audit_root = str(COOKBOOK_TMP_ROOT / "gateway_audit")
    if os.environ.get("OPENROUTER_API_KEY") and LiteLLMAdapter is not None:
        try:
            return (
                GatewayService(
                    GatewayConfig(
                        default_model=DEFAULT_MODEL,
                        audit=GatewayAuditConfig(persist_root=audit_root),
                    ),
                    provider_adapter=LiteLLMAdapter(),
                ),
                "litellm-live",
            )
        except Exception as exc:
            print(f"Falling back to NoopProviderAdapter: {exc}")
    elif LITELLM_IMPORT_ERROR is not None:
        print(f"LiteLLMAdapter unavailable; using noop gateway: {LITELLM_IMPORT_ERROR}")

    scripts = {
        "summarize_leaf_node": NoopScriptedResponse(
            output_json={
                "summary": "Deterministic notebook leaf summary.",
                "keywords": ["policy", "litigation", "revenue"],
            }
        ),
        "summarize_parent_node": NoopScriptedResponse(
            output_json={
                "summary": "Deterministic notebook parent summary.",
                "keywords": ["handbook", "policy", "summary"],
            }
        ),
    }
    return (
        GatewayService(
            GatewayConfig(
                default_model="noop-model",
                audit=GatewayAuditConfig(persist_root=audit_root),
            ),
            provider_adapter=NoopProviderAdapter(scripts),
        ),
        "noop-scripted",
    )


gateway, gateway_mode = build_demo_gateway()

show_json(
    "Notebook configuration",
    {
        "pdf_source_path": str(PDF_SOURCE_PATH),
        "markdown_source_path": str(MARKDOWN_SOURCE_PATH),
        "postgres_schema": POSTGRES_SCHEMA,
        "default_model": DEFAULT_MODEL,
        "gateway_mode": gateway_mode,
        "collection_id": COLLECTION_ID,
    },
)


## Section 4 — Acquisition (Spec 05: PDF and Markdown)


In [ ]:
acquisition_service = AcquisitionService(storage=pg_config)

pdf_acquisition = acquisition_service.acquire(
    AcquisitionRequest(
        source_path=str(PDF_SOURCE_PATH),
        acquisition_run_id=RUN_IDS["pdf_acquisition"],
        source_kind=SourceDocumentKind.PDF,
        provider_identity="native_pymupdf",
    )
)
markdown_acquisition = acquisition_service.acquire(
    AcquisitionRequest(
        source_path=str(MARKDOWN_SOURCE_PATH),
        acquisition_run_id=RUN_IDS["markdown_acquisition"],
        source_kind=SourceDocumentKind.MARKDOWN,
        provider_identity="markdown_native",
    )
)

pdf_acquisition_manifest_path = manifest_ref(
    run_type="acquisition",
    run_id=RUN_IDS["pdf_acquisition"],
    document_id=pdf_acquisition.document_id,
)
markdown_acquisition_manifest_path = manifest_ref(
    run_type="acquisition",
    run_id=RUN_IDS["markdown_acquisition"],
    document_id=markdown_acquisition.document_id,
)

show_json(
    "Acquisition manifests",
    {
        "pdf": {
            "document_id": pdf_acquisition.document_id,
            "page_count": pdf_acquisition.page_count,
            "selected_outline_source": pdf_acquisition.selected_outline_source,
            "manifest_ref": pdf_acquisition_manifest_path,
        },
        "markdown": {
            "document_id": markdown_acquisition.document_id,
            "page_count": markdown_acquisition.page_count,
            "selected_outline_source": markdown_acquisition.selected_outline_source,
            "manifest_ref": markdown_acquisition_manifest_path,
        },
    },
)


## Section 5 — Tree Build


In [ ]:
pdf_tree = build_tree(
    TreeBuildRequest(
        acquisition_manifest_path=pdf_acquisition_manifest_path,
        tree_run_id=RUN_IDS["pdf_tree"],
        summarize=False,
    ),
    storage=pg_config,
)
markdown_tree = build_tree(
    TreeBuildRequest(
        acquisition_manifest_path=markdown_acquisition_manifest_path,
        tree_run_id=RUN_IDS["markdown_tree"],
        summarize=True,
    ),
    gateway=gateway,
    storage=pg_config,
)

pdf_tree_manifest_path = manifest_ref(
    run_type="tree",
    run_id=RUN_IDS["pdf_tree"],
    document_id=pdf_tree.document_id,
)
markdown_tree_manifest_path = manifest_ref(
    run_type="tree",
    run_id=RUN_IDS["markdown_tree"],
    document_id=markdown_tree.document_id,
)

show_json(
    "Tree build summaries",
    {
        "pdf": {
            "tree_run_id": pdf_tree.tree_run_id,
            "committed_node_count": pdf_tree.committed_node_count,
            "node_card_titles": [card.title for card in load_node_cards_from_tree_manifest(pdf_tree)[:6]],
            "manifest_ref": pdf_tree_manifest_path,
        },
        "markdown": {
            "tree_run_id": markdown_tree.tree_run_id,
            "committed_node_count": markdown_tree.committed_node_count,
            "node_card_titles": [card.title for card in load_node_cards_from_tree_manifest(markdown_tree)[:6]],
            "manifest_ref": markdown_tree_manifest_path,
            "gateway_mode": gateway_mode,
        },
    },
)


## Section 6 — Tree Compaction (Spec 08)


In [ ]:
# max_children_per_node=2 is intentionally aggressive for demonstration purposes.
# The Markdown fixture has 5 sections under root, so this forces compaction to fire
# and produce synthetic serving nodes. Production values are typically 8-15.
markdown_compaction = TreeCompactionService(storage=pg_config).compact(
    TreeCompactionRequest(
        tree_manifest_path=markdown_tree_manifest_path,
        compaction_run_id=RUN_IDS["markdown_compaction"],
        settings=TreeCompactionSettings(max_children_per_node=2),
    )
)

compacted_tree = load_compacted_tree(
    markdown_compaction.compacted_tree_path,
    storage=pg_config,
)
compacted_mappings = load_compacted_node_mappings(
    markdown_compaction.node_mapping_path,
    storage=pg_config,
)
example_serving_node = next(
    (node for node in compacted_tree if "::compact::" in node.serving_node_id),
    compacted_tree[0],
)
expanded_canonical_ids = expand_serving_node_ids_to_canonical_node_ids(
    (example_serving_node.serving_node_id,),
    compacted_mappings,
)

show_json(
    "Compaction summary",
    {
        "tree_run_id": markdown_compaction.tree_run_id,
        "compaction_run_id": markdown_compaction.compaction_run_id,
        "canonical_node_count": markdown_tree.committed_node_count,
        "compacted_node_count": len(compacted_tree),
        "example_serving_node_id": example_serving_node.serving_node_id,
        "example_serving_title": example_serving_node.title,
        "example_canonical_node_ids": list(expanded_canonical_ids),
        "manifest_ref": manifest_ref(
            run_type="tree_compaction",
            run_id=RUN_IDS["markdown_compaction"],
            document_id=markdown_compaction.document_id,
        ),
    },
)

## Section 7 — Retrieval Corpus


In [ ]:
retrieval_builder = RetrievalCorpusBuilder(storage=pg_config)

pdf_retrieval = retrieval_builder.build(
    acquisition_manifest_path=pdf_acquisition_manifest_path,
    tree_manifest_path=pdf_tree_manifest_path,
    retrieval_run_id=RUN_IDS["pdf_retrieval"],
)
markdown_retrieval = retrieval_builder.build(
    acquisition_manifest_path=markdown_acquisition_manifest_path,
    tree_manifest_path=markdown_tree_manifest_path,
    retrieval_run_id=RUN_IDS["markdown_retrieval"],
)

markdown_corpus = load_retrieval_corpus(
    markdown_retrieval.corpus_path,
    storage=pg_config,
)
markdown_counts = Counter(unit.unit_type.value for unit in markdown_corpus.units)

show_json(
    "Retrieval corpus summary",
    {
        "pdf": {
            "document_id": pdf_retrieval.document_id,
            "unit_count": pdf_retrieval.unit_count,
            "manifest_ref": manifest_ref(
                run_type="retrieval",
                run_id=RUN_IDS["pdf_retrieval"],
                document_id=pdf_retrieval.document_id,
            ),
        },
        "markdown": {
            "document_id": markdown_retrieval.document_id,
            "unit_count": markdown_retrieval.unit_count,
            "counts_by_type": dict(sorted(markdown_counts.items())),
            "corpus_ref": markdown_retrieval.corpus_path,
            "manifest_ref": manifest_ref(
                run_type="retrieval",
                run_id=RUN_IDS["markdown_retrieval"],
                document_id=markdown_retrieval.document_id,
            ),
        },
    },
)


## Section 8 — Document Descriptions (Spec 01)


In [ ]:
description_builder = DocumentDescriptionBuilder(storage=pg_config)

pdf_description_manifest = description_builder.build(
    DocumentDescriptionRequest(
        acquisition_manifest_path=pdf_acquisition_manifest_path,
        tree_manifest_path=pdf_tree_manifest_path,
        description_run_id=RUN_IDS["pdf_description"],
    )
)
markdown_description_manifest = description_builder.build(
    DocumentDescriptionRequest(
        acquisition_manifest_path=markdown_acquisition_manifest_path,
        tree_manifest_path=markdown_tree_manifest_path,
        description_run_id=RUN_IDS["markdown_description"],
    )
)

pdf_description_manifest_ref = manifest_ref(
    run_type="document_description",
    run_id=RUN_IDS["pdf_description"],
    document_id=pdf_description_manifest.document_id,
)
markdown_description_manifest_ref = manifest_ref(
    run_type="document_description",
    run_id=RUN_IDS["markdown_description"],
    document_id=markdown_description_manifest.document_id,
)

pdf_description_manifest_loaded = load_document_description_manifest(
    pdf_description_manifest_ref,
    storage=pg_config,
)
markdown_description_manifest_loaded = load_document_description_manifest(
    markdown_description_manifest_ref,
    storage=pg_config,
)
pdf_description = load_document_description(
    pdf_description_manifest_loaded.description_path,
    storage=pg_config,
)
markdown_description = load_document_description(
    markdown_description_manifest_loaded.description_path,
    storage=pg_config,
)

show_json(
    "Document descriptions",
    {
        "pdf": {
            "description_method": pdf_description.description_method,
            "description_text": pdf_description.description_text,
            "source_node_ids": list(pdf_description.source_node_ids),
            "manifest_ref": pdf_description_manifest_ref,
        },
        "markdown": {
            "description_method": markdown_description.description_method,
            "description_text": markdown_description.description_text,
            "source_node_ids": list(markdown_description.source_node_ids),
            "manifest_ref": markdown_description_manifest_ref,
        },
    },
)


## Section 9 — Collection Selection Before Retrieval

Three selection strategies narrow a multi-document collection before retrieval:
- **Spec 02 — Metadata Selection**: deterministic attribute filters (EQ, CONTAINS)
- **Spec 03 — Description Selection**: ranks documents by description similarity to a query
- **Spec 04 — Semantic Prefilter**: matches against multi-field semantic proxies built from descriptions and node titles

### 9a — Metadata Selection (Spec 02)

In [ ]:
metadata_records = (
    DocumentMetadataRecord(
        document_id=pdf_retrieval.document_id,
        display_name="Born Digital Outline PDF",
        attributes={
            "source_kind": "pdf",
            "company": "NullVector",
            "year": 2024,
            "topic": "outline demo and born-digital reference",
        },
    ),
    DocumentMetadataRecord(
        document_id=markdown_retrieval.document_id,
        display_name="Operating Handbook Markdown",
        attributes={
            "source_kind": "markdown",
            "company": "Policy Labs",
            "year": 2025,
            "topic": "operating handbook policies and litigation deadlines",
        },
    ),
)

metadata_response = MetadataSelectionService(storage=pg_config).select(
    MetadataSelectionRequest(
        collection_id=COLLECTION_ID,
        selection_run_id=RUN_IDS["metadata_selection"],
        allowed_fields=("source_kind", "company", "year", "topic"),
        metadata_records=metadata_records,
        plan=MetadataSelectionPlan(
            raw_query="markdown policy handbook",
            normalized_query="markdown policy handbook",
            clauses=(
                DocumentFilterClause(
                    field="source_kind",
                    operator=DocumentFilterOperator.EQ,
                    value="markdown",
                ),
                DocumentFilterClause(
                    field="topic",
                    operator=DocumentFilterOperator.CONTAINS,
                    value="policy",
                ),
            ),
        ),
        limit=2,
    )
)

show_json(
    "Metadata selection (Spec 02)",
    {
        "candidate_ids": [candidate.document_id for candidate in metadata_response.candidates],
        "matched_metadata": [candidate.matched_metadata for candidate in metadata_response.candidates],
        "results_ref": metadata_response.selection_results_path,
    },
)

### 9b — Description Selection (Spec 03)

In [ ]:
description_records = (
    DocumentDescriptionRecord(
        document_id=pdf_retrieval.document_id,
        display_name="Born Digital Outline PDF",
        description_text=pdf_description.description_text,
        description_manifest_path=pdf_description_manifest_ref,
    ),
    DocumentDescriptionRecord(
        document_id=markdown_retrieval.document_id,
        display_name="Operating Handbook Markdown",
        description_text=markdown_description.description_text,
        description_manifest_path=markdown_description_manifest_ref,
    ),
)
description_selection_response = DescriptionSelectionService(storage=pg_config).select(
    DescriptionSelectionRequest(
        collection_id=COLLECTION_ID,
        selection_run_id=RUN_IDS["description_selection"],
        query="Which document focuses on operating handbook policies and deadlines?",
        descriptions=description_records,
        limit=2,
    )
)

show_json(
    "Description selection (Spec 03)",
    {
        "candidate_ids": [candidate.document_id for candidate in description_selection_response.candidates],
        "scores": [candidate.score for candidate in description_selection_response.candidates],
        "results_ref": description_selection_response.selection_results_path,
    },
)

### 9c — Semantic Prefilter (Spec 04)

In [ ]:
semantic_proxy_sources = (
    DocumentSemanticProxySource(
        document_id=pdf_retrieval.document_id,
        display_name="Born Digital Outline PDF",
        description_manifest_path=pdf_description_manifest_ref,
        tree_manifest_path=pdf_tree_manifest_path,
    ),
    DocumentSemanticProxySource(
        document_id=markdown_retrieval.document_id,
        display_name="Operating Handbook Markdown",
        description_manifest_path=markdown_description_manifest_ref,
        tree_manifest_path=markdown_tree_manifest_path,
    ),
)
semantic_proxies = DocumentSemanticProxyBuilder(storage=pg_config).build(
    semantic_proxy_sources,
)
semantic_prefilter_response = SemanticPrefilterService(storage=pg_config).select(
    DocumentPrefilterRequest(
        collection_id=COLLECTION_ID,
        selection_run_id=RUN_IDS["semantic_prefilter"],
        query="litigation policies and case deadlines",
        proxies=semantic_proxies,
        limit=2,
    )
)

show_json(
    "Semantic prefilter (Spec 04)",
    {
        "document_ids": [hit.document_id for hit in semantic_prefilter_response.hits],
        "matched_proxy_fields": [list(hit.matched_proxy_fields) for hit in semantic_prefilter_response.hits],
        "results_ref": semantic_prefilter_response.semantic_prefilter_results_path,
    },
)

## Section 10 — Tree Search (Spec 06)


In [ ]:
planner = QueryPlanner()
ranker = RetrievalRanker()
retrieval_service = RetrievalService(planner, ranker, storage=pg_config)
tree_search_service = TreeSearchService(planner, retrieval_service, storage=pg_config)
markdown_titles_by_id = node_title_map(markdown_tree)

tree_search_response = tree_search_service.search(
    TreeSearchRequest(
        query="litigation policies",
        tree_manifest_path=markdown_tree_manifest_path,
        retrieval_manifest_path=manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["markdown_retrieval"],
            document_id=markdown_retrieval.document_id,
        ),
        search_run_id=RUN_IDS["tree_search"],
        max_selected_nodes=1,
        retrieval_limit=5,
    )
)

show_json(
    "Tree search response",
    {
        "search_mode": tree_search_response.search_mode,
        "selected_node_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in tree_search_response.selected_nodes
        ],
        "trace": [step.model_dump(mode="json") for step in tree_search_response.trace],
        "retrieval_hit_ids": [hit.unit.unit_id for hit in tree_search_response.retrieval_hits],
        "results_ref": tree_search_response.results_path,
    },
)


## Section 11 — Preference-Aware Tree Search (Spec 07)


In [ ]:
preference_tree_search_service = PreferenceAwareTreeSearchService(
    planner,
    retrieval_service,
    storage=pg_config,
)

base_policy_tree_search = tree_search_service.search(
    TreeSearchRequest(
        query="policies",
        tree_manifest_path=markdown_tree_manifest_path,
        retrieval_manifest_path=manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["markdown_retrieval"],
            document_id=markdown_retrieval.document_id,
        ),
        search_run_id=f"{RUN_IDS['preference_tree_search']}-base",
        max_selected_nodes=1,
        retrieval_limit=5,
    )
)
preference_tree_search_response = preference_tree_search_service.search(
    PreferenceAwareTreeSearchRequest(
        query="policies",
        tree_manifest_path=markdown_tree_manifest_path,
        retrieval_manifest_path=manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["markdown_retrieval"],
            document_id=markdown_retrieval.document_id,
        ),
        search_run_id=RUN_IDS["preference_tree_search"],
        max_selected_nodes=1,
        retrieval_limit=5,
        preference_snippets=(
            PreferenceSnippet(
                preference_id="prefer-litigation",
                scope=PreferenceScope.USER,
                text="Prefer litigation policies and case deadlines when policy sections tie.",
                priority=5,
            ),
        ),
    )
)

show_json(
    "Preference-aware tree search",
    {
        "base_selected_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in base_policy_tree_search.selected_nodes
        ],
        "preference_selected_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in preference_tree_search_response.selected_nodes
        ],
        "preference_selection": preference_tree_search_response.preference_selection.model_dump(mode="json"),
        "trace": [step.model_dump(mode="json") for step in preference_tree_search_response.trace],
        "results_ref": preference_tree_search_response.results_path,
    },
)


## Section 12 — Retrieval QA (self-contained NullVector path)


In [ ]:
# RetrievalService was constructed with storage=pg_config (cell 20) to enable
# document-id-based search from Postgres. Here we pass corpus= directly (already
# loaded in-memory from cell 14), so the storage kwarg is not exercised in this path.
qa_query = "What do the litigation policies say about deadlines?"
retrieval_hits = retrieval_service.search(
    corpus=markdown_corpus,
    query=qa_query,
    limit=5,
)
qa_service = RetrievalQAService(retrieval_service)
qa_response = qa_service.answer(
    corpus=markdown_corpus,
    query=qa_query,
    limit=5,
)

show_json(
    "Retrieval QA",
    {
        "query": qa_query,
        "retrieval_hit_ids": [hit.unit.unit_id for hit in retrieval_hits],
        "answer_mode": qa_response.answer_mode,
        "answer": qa_response.answer,
        "citations": [citation.model_dump(mode="json") for citation in qa_response.citations],
    },
)

## Section 13 — Quickstart CLI (Spec 09)

The quickstart CLI stays intentionally thin: acquire, build a tree, and optionally build retrieval artifacts. The equivalent Postgres-backed command for the Markdown fixture created above is:

```bash
export NULLVECTOR_POSTGRES_CONNINFO='postgresql://REDACTED_DB_CRED@localhost:5432/nullvector'
python scripts/nullvector_quickstart.py           --source-path cookbook/_tmp/postgres_unified/spec05_operating_handbook.md           --storage-backend postgres           --pg-conninfo "$NULLVECTOR_POSTGRES_CONNINFO"           --build-retrieval           --print-tree-summary
```

Use the same command shape with the committed PDF fixture when you want the edge-only CLI path instead of the library walkthrough shown in this notebook.


In [ ]:
# Uncomment to run the quickstart CLI directly from this notebook:
# !python scripts/nullvector_quickstart.py \
#     --source-path cookbook/_tmp/postgres_unified/spec05_operating_handbook.md \
#     --storage-backend postgres \
#     --pg-conninfo "$NULLVECTOR_POSTGRES_CONNINFO" \
#     --build-retrieval \
#     --print-tree-summary

## Section 14 — Results Inspection


In [ ]:
summary = {
    "documents": {
        "pdf_document_id": pdf_acquisition.document_id,
        "markdown_document_id": markdown_acquisition.document_id,
    },
    "manifest_refs": {
        "pdf_acquisition": pdf_acquisition_manifest_path,
        "pdf_tree": pdf_tree_manifest_path,
        "pdf_retrieval": manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["pdf_retrieval"],
            document_id=pdf_retrieval.document_id,
        ),
        "pdf_description": pdf_description_manifest_ref,
        "markdown_acquisition": markdown_acquisition_manifest_path,
        "markdown_tree": markdown_tree_manifest_path,
        "markdown_compaction": manifest_ref(
            run_type="tree_compaction",
            run_id=RUN_IDS["markdown_compaction"],
            document_id=markdown_compaction.document_id,
        ),
        "markdown_retrieval": manifest_ref(
            run_type="retrieval",
            run_id=RUN_IDS["markdown_retrieval"],
            document_id=markdown_retrieval.document_id,
        ),
        "markdown_description": markdown_description_manifest_ref,
    },
    "selection_winners": {
        "metadata": [candidate.document_id for candidate in metadata_response.candidates],
        "description": [candidate.document_id for candidate in description_selection_response.candidates],
        "semantic_prefilter": [hit.document_id for hit in semantic_prefilter_response.hits],
    },
    "tree_search": {
        "selected_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in tree_search_response.selected_nodes
        ],
        "preference_selected_titles": [
            markdown_titles_by_id.get(candidate.node_id, candidate.node_id)
            for candidate in preference_tree_search_response.selected_nodes
        ],
    },
    "qa_answer": qa_response.answer,
}
show_json("Unified Postgres cookbook summary", summary)


## Notes

- This notebook is designed for structural correctness and manual execution against a live PostgreSQL database; it is not executed in CI.
- The Markdown source is generated locally inside `cookbook/_tmp/postgres_unified/` so Spec 05 can be demonstrated alongside the committed PDF fixture. This directory is gitignored — no accidental commits of generated fixtures.
- The collection-selection block is split into three subsections (9a, 9b, 9c) corresponding to Specs 02, 03, and 04 so each strategy's inputs and outputs are independently visible.
- The quickstart CLI remains intentionally narrower than the library walkthrough: it covers acquisition, tree build, and optional retrieval only. A commented-out executable cell is provided for copy-paste convenience.